# 03 - Modelisation et Evaluation

Objectif: entrainer un Random Forest pour predire la gravite (binaire) et sauvegarder le modele.

Sorties attendues:
- model/modele_gravite_rf.joblib
- model/colonnes_modele.joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
from IPython.display import display


In [ ]:
df = pd.read_csv('data/accidents_scored.csv', low_memory=False)

# Rattachement de la cible grave depuis usagers
csv_params = {'sep': ',', 'encoding': 'latin1', 'low_memory': False}
df_gravite = pd.read_csv('data/usagers.csv', usecols=['num_acc', 'grav'], **csv_params)
df_gravite['grav'] = pd.to_numeric(df_gravite['grav'], errors='coerce')
df_gravite = df_gravite.dropna(subset=['grav'])
df_gravite['grave'] = df_gravite['grav'].isin([2, 3]).astype(int)

target = df_gravite.groupby('num_acc')['grave'].max().reset_index()
df = df.merge(target, on='num_acc', how='left')
df['grave'] = df['grave'].fillna(0).astype(int)

print(df['grave'].value_counts(normalize=True))


In [ ]:
FEATURES = ['mois', 'jour', 'lum', 'agg', 'int', 'atm', 'col', 'catr', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'env1', 'vma', 'nb_usagers', 'nb_pietons', 'pct_pietons', 'heure', 'anomaly_score']

X = df[FEATURES].copy()
y = df['grave']

X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape, 'X_test:', X_test.shape)


In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

model_rf.fit(X_train, y_train)


In [ ]:
y_pred = model_rf.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Leger/Indemne', 'Grave'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Matrice de confusion')
plt.show()


In [ ]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importances.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 - Importance des variables')
plt.show()

display(importances.head(10))


In [ ]:
joblib.dump(model_rf, 'model/modele_gravite_rf.joblib')
joblib.dump(FEATURES, 'model/colonnes_modele.joblib')
print('Modele et colonnes sauvegardes')
